# Agregações de crescimento e receita

Responde à trilha **Crescimento e Receita** do Tech Challenge:
1. Evolução mensal de pedidos, receita e ticket médio.
2. Participação por categoria e por UF.
3. Produtos e sellers com melhor desempenho.

As tabelas agregadas são salvas em `data/processed/` com o prefixo `agg_`, para uso no relatório e na apresentação.

## Bibliotecas

In [ ]:
%run ../src/librarys.ipynb

## Carga da camada processed

In [ ]:
orders_enriched = pd.read_csv("../data/processed/orders_enriched.csv", parse_dates=["order_purchase_timestamp"])
order_items_category = pd.read_csv("../data/processed/order_items_category.csv", parse_dates=["order_purchase_timestamp"])
sellers = pd.read_csv("../data/processed/sellers.csv", dtype={"seller_zip_code_prefix": str})
customers = pd.read_csv("../data/processed/customers.csv", dtype={"customer_zip_code_prefix": str})

## Regras das agregações

Seguem as regras definidas no [relatório de qualidade](../docs/relatorio-de-qualidade.md):

- **Somente pedidos entregues (`DELIVERED`):** pedidos cancelados, indisponíveis ou em andamento não representam venda realizada.
- **Período de jan/2017 a ago/2018:** os meses fora desse intervalo têm volume muito baixo (início da operação e corte da extração) e criariam quedas artificiais nos gráficos.
- **Receita = preço + frete:** valor total pago pelo cliente (GMV). É o mesmo valor de `order_total`.
- **Ticket médio = receita ÷ número de pedidos.**

In [ ]:
pedidos = orders_enriched[
    (orders_enriched["order_status"] == "DELIVERED")
    & (orders_enriched["order_purchase_timestamp"] >= "2017-01-01")
    & (orders_enriched["order_purchase_timestamp"] < "2018-09-01")
]

itens = order_items_category[
    (order_items_category["order_status"] == "DELIVERED")
    & (order_items_category["order_purchase_timestamp"] >= "2017-01-01")
    & (order_items_category["order_purchase_timestamp"] < "2018-09-01")
].copy()
itens["item_total"] = itens["price"] + itens["freight_value"]

print("Pedidos considerados:", len(pedidos), "de", len(orders_enriched))
print("Itens considerados:", len(itens), "de", len(order_items_category))

## Indicadores gerais do período

Visão consolidada para abrir o relatório. Clientes são contados pelo `customer_unique_id`, que identifica a pessoa (o `customer_id` muda a cada pedido).

In [ ]:
clientes = pedidos.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")

receita_total = pedidos["order_total"].sum()
total_pedidos = pedidos["order_id"].nunique()

print(f"Pedidos: {total_pedidos:,}")
print(f"Receita: R$ {receita_total:,.2f}")
print(f"Ticket médio: R$ {receita_total / total_pedidos:,.2f}")
print(f"Clientes únicos: {clientes['customer_unique_id'].nunique():,}")
print(f"Sellers ativos: {itens['seller_id'].nunique():,}")
print(f"Produtos vendidos: {itens['product_id'].nunique():,}")

## 1. Evolução mensal

Agrupa os pedidos por mês da compra e calcula quantidade de pedidos, receita e ticket médio. O crescimento mensal compara cada mês com o anterior.

In [ ]:
agg_mensal = pedidos.groupby("purchase_year_month").agg(
    pedidos=("order_id", "count"),
    receita=("order_total", "sum"),
).reset_index()

agg_mensal["ticket_medio"] = agg_mensal["receita"] / agg_mensal["pedidos"]
agg_mensal["crescimento_receita_pct"] = agg_mensal["receita"].pct_change() * 100

agg_mensal

Pedidos, receita e ticket médio lado a lado. Pedidos e receita andam juntos; o ticket médio mostra se o crescimento vem de mais pedidos ou de pedidos mais caros.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)

axes[0].plot(agg_mensal["purchase_year_month"], agg_mensal["pedidos"], marker="o")
axes[0].set_title("Pedidos por mês")

axes[1].plot(agg_mensal["purchase_year_month"], agg_mensal["receita"] / 1_000_000, marker="o", color="tab:green")
axes[1].set_title("Receita por mês (R$ milhões)")

axes[2].plot(agg_mensal["purchase_year_month"], agg_mensal["ticket_medio"], marker="o", color="tab:orange")
axes[2].set_title("Ticket médio por mês (R$)")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Comparação entre os 8 primeiros meses de 2017 e de 2018 (mesmo período nos dois anos, para uma comparação justa).

In [ ]:
pedidos_ano = pedidos[pedidos["order_purchase_timestamp"].dt.month <= 8].copy()
pedidos_ano["ano"] = pedidos_ano["order_purchase_timestamp"].dt.year

agg_ano = pedidos_ano.groupby("ano").agg(
    pedidos=("order_id", "count"),
    receita=("order_total", "sum"),
)
agg_ano["ticket_medio"] = agg_ano["receita"] / agg_ano["pedidos"]

agg_ano.loc["crescimento_%"] = (agg_ano.loc[2018] / agg_ano.loc[2017] - 1) * 100
agg_ano

## 2. Participação por categoria

Receita, pedidos e itens vendidos por categoria, com a participação de cada uma na receita total. A tabela é no nível de item, porque um pedido pode ter produtos de categorias diferentes.

In [ ]:
agg_categoria = itens.groupby(["product_category_name", "product_category_name_english"]).agg(
    pedidos=("order_id", "nunique"),
    itens=("order_id", "count"),
    receita=("item_total", "sum"),
).reset_index()

agg_categoria["participacao_receita_pct"] = agg_categoria["receita"] / agg_categoria["receita"].sum() * 100
agg_categoria = agg_categoria.sort_values("receita", ascending=False)

agg_categoria.head(15)

Concentração da receita: quantas categorias respondem pela maior parte do faturamento.

In [ ]:
participacao_acumulada = agg_categoria["participacao_receita_pct"].cumsum()

print("Total de categorias:", len(agg_categoria))
print(f"Top 5 categorias: {participacao_acumulada.iloc[4]:.1f}% da receita")
print(f"Top 10 categorias: {participacao_acumulada.iloc[9]:.1f}% da receita")
print("Categorias para chegar a 80% da receita:", (participacao_acumulada < 80).sum() + 1)

In [ ]:
top15_categorias = agg_categoria.head(15).sort_values("participacao_receita_pct")

top15_categorias.plot(kind="barh", x="product_category_name", y="participacao_receita_pct", legend=False)
plt.title("Participação na receita - 15 maiores categorias (%)")
plt.xlabel("% da receita")
plt.ylabel("")
plt.show()

## 3. Participação por UF

Receita, pedidos e ticket médio por estado do cliente, com a participação de cada UF na receita total.

In [ ]:
agg_uf = pedidos.groupby("customer_state").agg(
    pedidos=("order_id", "count"),
    receita=("order_total", "sum"),
).reset_index()

agg_uf["ticket_medio"] = agg_uf["receita"] / agg_uf["pedidos"]
agg_uf["participacao_receita_pct"] = agg_uf["receita"] / agg_uf["receita"].sum() * 100
agg_uf = agg_uf.sort_values("receita", ascending=False)

agg_uf

In [ ]:
uf_ordenado = agg_uf.sort_values("participacao_receita_pct")

uf_ordenado.plot(kind="barh", x="customer_state", y="participacao_receita_pct", legend=False, figsize=(10, 9))
plt.title("Participação na receita por UF (%)")
plt.xlabel("% da receita")
plt.ylabel("")
plt.show()

Ticket médio por UF. Estados mais distantes do Sudeste tendem a ter ticket maior, em parte pelo frete mais caro.

In [ ]:
uf_ticket = agg_uf.sort_values("ticket_medio")

uf_ticket.plot(kind="barh", x="customer_state", y="ticket_medio", legend=False, figsize=(10, 9), color="tab:orange")
plt.title("Ticket médio por UF (R$)")
plt.xlabel("R$")
plt.ylabel("")
plt.show()

## 4. Top produtos

Os 10 produtos com maior receita, com a quantidade de itens vendidos e o preço médio.

In [ ]:
agg_produto = itens.groupby(["product_id", "product_category_name"]).agg(
    pedidos=("order_id", "nunique"),
    itens=("order_id", "count"),
    receita=("item_total", "sum"),
    preco_medio=("price", "mean"),
).reset_index()

agg_produto["participacao_receita_pct"] = agg_produto["receita"] / agg_produto["receita"].sum() * 100
top_produtos = agg_produto.sort_values("receita", ascending=False).head(10)

top_produtos

Os 10 produtos mais vendidos em quantidade. Comparar com o ranking por receita mostra a diferença entre produtos de alto giro e produtos de alto valor.

In [ ]:
agg_produto.sort_values("itens", ascending=False).head(10)

## 5. Top sellers

Os 10 vendedores com maior receita. Além do volume, trazemos avaliação média e percentual de atraso, para identificar quem vende muito **e** entrega bem.

A avaliação e o atraso são do pedido; por isso, antes de calcular a média, ficamos com um registro por pedido e seller (um pedido com 3 itens do mesmo seller conta uma vez).

In [ ]:
agg_seller = itens.groupby("seller_id").agg(
    pedidos=("order_id", "nunique"),
    itens=("order_id", "count"),
    receita=("item_total", "sum"),
).reset_index()

pedido_seller = itens.drop_duplicates(["order_id", "seller_id"])
qualidade_seller = pedido_seller.groupby("seller_id").agg(
    review_medio=("review_score", "mean"),
    pct_atraso=("is_late", "mean"),
).reset_index()
qualidade_seller["pct_atraso"] = qualidade_seller["pct_atraso"] * 100

agg_seller = agg_seller.merge(qualidade_seller, on="seller_id", how="left")
agg_seller = agg_seller.merge(sellers[["seller_id", "seller_city", "seller_state"]], on="seller_id", how="left")
agg_seller["participacao_receita_pct"] = agg_seller["receita"] / agg_seller["receita"].sum() * 100

top_sellers = agg_seller.sort_values("receita", ascending=False).head(10)
top_sellers

Concentração da receita entre os sellers.

In [ ]:
participacao_seller = agg_seller.sort_values("receita", ascending=False)["participacao_receita_pct"].cumsum()

print("Sellers ativos:", len(agg_seller))
print(f"Top 10 sellers: {participacao_seller.iloc[9]:.1f}% da receita")
print(f"Top 100 sellers: {participacao_seller.iloc[99]:.1f}% da receita")

Média geral para comparação com o top 10.

In [ ]:
print(f"Review médio geral: {pedido_seller['review_score'].mean():.2f}")
print(f"% de atraso geral: {pedido_seller['is_late'].mean() * 100:.1f}%")

## Conferência

A receita total deve ser a mesma em todas as agregações. Se algum valor diferir, algum filtro ou join está inconsistente.

In [ ]:
print(f"Mensal:    R$ {agg_mensal['receita'].sum():,.2f}")
print(f"UF:        R$ {agg_uf['receita'].sum():,.2f}")
print(f"Categoria: R$ {agg_categoria['receita'].sum():,.2f}")
print(f"Produto:   R$ {agg_produto['receita'].sum():,.2f}")
print(f"Seller:    R$ {agg_seller['receita'].sum():,.2f}")

## Salvar

Grava as agregações em `data/processed/`.

In [ ]:
agg_mensal.to_csv("../data/processed/agg_mensal.csv", index=False)
agg_categoria.to_csv("../data/processed/agg_categoria.csv", index=False)
agg_uf.to_csv("../data/processed/agg_uf.csv", index=False)
agg_produto.to_csv("../data/processed/agg_produto.csv", index=False)
agg_seller.to_csv("../data/processed/agg_seller.csv", index=False)